# LAB exam assignment 1


# Explorative Data Analysis
In this notebook, we perform a comprehensive audit of `marketing_campaign.csv `
We will address missing values and outliers, and justify our cleaning strategy based on the feature distributions and the risk of introducing bias into downstream models.
Then, we will perform a descriptive analysis on observed trends from a customer perspective.

We will use the marketing_campaign.csv dataset.

# Data Dictionary

Quick reference for every column in the `marketing_campaign.csv` dataset.

---

## `marketing_campaign.csv` — Customer Profile

| Column | Type | Description |
|---|---|---|
| `ID` | integer | Unique identifier for each customer |
| `Year_Birth` | integer | Customer's birth year |
| `Education` | string | Customer's education level (`Basic`, `2n Cycle`, `Graduation`, `Master`, `PhD`) |
| `Marital_Status` | string | Customer's marital status (`Single`, `Married`, `Together`, `Divorced`, `Widow`) |
| `Income` | float | Customer's yearly household income in euros |
| `Kidhome` | integer | Number of small children in the customer's household |
| `Teenhome` | integer | Number of teenagers in the customer's household |
| `Dt_Customer` | string (date) | Date of customer's enrollment with the company |
| `Recency` | integer | Number of days since the customer's last purchase |

---

## `marketing_campaign.csv` — Spending Behaviour

| Column | Type | Description |
|---|---|---|
| `MntWines` | float | Amount spent on wine in the last 2 years (€) |
| `MntFruits` | float | Amount spent on fruits in the last 2 years (€) |
| `MntMeatProducts` | float | Amount spent on meat in the last 2 years (€) |
| `MntFishProducts` | float | Amount spent on fish in the last 2 years (€) |
| `MntSweetProducts` | float | Amount spent on sweets in the last 2 years (€) |
| `MntGoldProds` | float | Amount spent on gold products in the last 2 years (€) |

---

## `marketing_campaign.csv` — Purchase Channels

| Column | Type | Description |
|---|---|---|
| `NumDealsPurchases` | integer | Number of purchases made with a discount |
| `NumWebPurchases` | integer | Number of purchases made through the company's website |
| `NumCatalogPurchases` | integer | Number of purchases made using a catalogue |
| `NumStorePurchases` | integer | Number of purchases made directly in stores |
| `NumWebVisitsMonth` | integer | Number of visits to the company's website in the last month |

---

## `marketing_campaign.csv` — Campaign Responses

| Column | Type | Description |
|---|---|---|
| `AcceptedCmp1` | integer | `1` if customer accepted the offer in the 1st campaign, `0` otherwise |
| `AcceptedCmp2` | integer | `1` if customer accepted the offer in the 2nd campaign, `0` otherwise |
| `AcceptedCmp3` | integer | `1` if customer accepted the offer in the 3rd campaign, `0` otherwise |
| `AcceptedCmp4` | integer | `1` if customer accepted the offer in the 4th campaign, `0` otherwise |
| `AcceptedCmp5` | integer | `1` if customer accepted the offer in the 5th campaign, `0` otherwise |
| `Complain` | integer | `1` if customer complained in the last 2 years, `0` otherwise |
| `Response` | integer | `1` if customer accepted the offer in the **last** campaign, `0` otherwise |

Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load data
df = pd.read_csv('marketing_campaign.csv', sep=';')

print("Marketing Data shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Marketing Data shape: (2240, 27)

Column names:
['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response']


## 1. Comprehensive Audit of the Data
We will examine the variable types, missing values, duplicates, and basic descriptive statistics for both datasets.


In [3]:
# --- Marketing Campaign Dataset Info ---
print("--- Marketing Campaign Dataset Info ---")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

--- Marketing Campaign Dataset Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15

In [4]:
# Verify all categorical column values
cat_cols = ['Education', 'Marital_Status']

for col in cat_cols:
    print(f"\n{col} — unique values:")
    print(df[col].unique())
    print(f"  Value counts:\n{df[col].value_counts()}")


Education — unique values:
['Graduation' 'PhD' 'Master' 'Basic' '2n Cycle']
  Value counts:
Education
Graduation    1127
PhD            486
Master         370
2n Cycle       203
Basic           54
Name: count, dtype: int64

Marital_Status — unique values:
['Single' 'Together' 'Married' 'Divorced' 'Widow' 'Alone' 'Absurd' 'YOLO']
  Value counts:
Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


### Analysis of the Audit:

**Marketing Campaign Dataset:**
- `Income` is the only column with missing values: **24 null entries** (~1.07% of the data)
- `Dt_Customer` is stored as `object` (string) — it needs to be converted to `datetime` to be usable for time-based analysis
- `Education` and `Marital_Status` are stored as `object` (string) — correct type, but inspection reveals **invalid category values** in `Marital_Status`: `'Absurd'`, `'YOLO'` (2 entries each) and `'Alone'` (3 entries, duplicate of `'Single'`) that need to be handled
- All spending and purchase columns are correctly typed as `int64`
- No duplicate rows detected

## 2. Addressing Missing Values and Outliers

First, let's cast incorrect data types to see if structural errors turn into missing values.
We start from `Dt_Customer` since we know it should be a date but is stored as object.
We first inspect the raw values to identify the format, then convert accordingly.

In [5]:
# Check the actual date format before converting
print(df['Dt_Customer'].head(5))

0    2012-09-04
1    2014-03-08
2    2013-08-21
3    2014-02-10
4    2014-01-19
Name: Dt_Customer, dtype: object


Output confirms format is YYYY-MM-DD , so we use format='%Y-%m-%d'

### Deriving Customer Age from `Dt_Customer`

The `Dt_Customer` column records the date on which each customer was enrolled in the company. Rather than keeping this as a raw date, it is more analytically useful to derive the **age of the customer** at the time they were active in the dataset — since age is a meaningful socio-demographic feature for segmentation and spending analysis.

To compute age, we need a reference date to subtract from `Year_Birth`. Two approaches are defensible here. The first would be to use today's date, which is simple but problematic: it would make customers appear significantly older than they actually were when they were making their purchasing decisions, distorting any analysis that links age to spending behaviour. The second — and the one we adopt here — is to use the **most recent enrollment date present in the dataset** as a proxy for the data collection cutoff. This anchors Age to the period the data actually covers, and has the advantage of being derived directly from the data itself rather than relying on an external, arbitrary date.

In our case, the most recent enrollment date is **June 2014**, so we use **2014 as the reference year**.

In [ ]:
# Convert to datetime
df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%Y-%m-%d')

# We don't know the exact data collection date.
# We use the most recent record in the dataset as a reference point,
# so that Age reflects how old customers were during the period the data covers.
reference_year = df['Dt_Customer'].dt.year.max()
print("Reference year used:", reference_year)



Reference year used: 2014


In [ ]:
# Compute Age relative to that reference year
df['Age'] = reference_year - df['Year_Birth']

# Verify the conversion worked — check dtype, not missing values
print("Dt_Customer dtype after conversion:", df['Dt_Customer'].dtype)
print("Age dtype:", df['Age'].dtype)
print("\nSample:")
print(df[['Dt_Customer', 'Age']].head(3))

Dt_Customer dtype after conversion: datetime64[ns]
Age dtype: int64

Sample:
  Dt_Customer  Age
0  2012-09-04   57
1  2014-03-08   60
2  2013-08-21   49
